In [ ]:
import os

base = r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_ENTIRE"

train_images = len(os.listdir(os.path.join(base,"images","train")))
train_labels = len(os.listdir(os.path.join(base,"labels","train")))

val_images = len(os.listdir(os.path.join(base,"images","val")))
val_labels = len(os.listdir(os.path.join(base,"labels","val")))

print("Train Images :", train_images)
print("Train Labels :", train_labels)

print("Val Images :", val_images)
print("Val Labels :", val_labels)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8l.pt")

model.train(
    data=r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_ENTIRE\acdc_entire.yaml",
    epochs=100,
    imgsz=640,
    batch=4,
    workers=4,
    device=0,
    cache=False,
    amp=True,
    project="ACDC_Entire_Project",
    name="train_yolov8l_entire"
)

In [ ]:
import os
import shutil
import pandas as pd

# 1. Configuration
CONDITION = "entire"
PROJECT_DIR = r"C:\Users\Varis\runs\detect\ACDC_ENTIRE_Project\train_yolov8l_entire-4"
BACKUP_DIR = r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_ENTIRE"

# 2. Ensure backup directory exists
os.makedirs(BACKUP_DIR, exist_ok=True)

# 3. Files to backup
files_to_copy = [
    "results.csv", "results.png", "confusion_matrix.png", 
    "confusion_matrix_normalized.png", "PR_curve.png", "P_curve.png", 
    "R_curve.png", "F1_curve.png", "labels.jpg", 
    "labels_correlogram.jpg", "args.yaml"
]

# Copy Weights (best.pt and last.pt)
shutil.copy2(os.path.join(PROJECT_DIR, "weights", "best.pt"), BACKUP_DIR)
shutil.copy2(os.path.join(PROJECT_DIR, "weights", "last.pt"), BACKUP_DIR)

# Copy Diagnostic Files
for file in files_to_copy:
    src_path = os.path.join(PROJECT_DIR, file)
    if os.path.exists(src_path):
        shutil.copy2(src_path, BACKUP_DIR)
    else:
        print(f"Warning: {file} not found in project directory.")

# 4. Create Performance Summary CSV
results_df = pd.read_csv(os.path.join(PROJECT_DIR, "results.csv"))
results_df.columns = results_df.columns.str.strip()
summary_df = pd.DataFrame([{
    "Condition": CONDITION.upper(),
    "Max_mAP50": results_df['metrics/mAP50(B)'].max(),
    "Max_mAP50-95": results_df['metrics/mAP50-95(B)'].max()
}])
summary_df.to_csv(os.path.join(BACKUP_DIR, "performance_summary.csv"), index=False)

# 5. Create Dissertation Info File
with open(os.path.join(BACKUP_DIR, "training_info.txt"), "w") as f:
    f.write(f"Dissertation Project Backup: {CONDITION.upper()}\n")
    f.write(f"Path: {PROJECT_DIR}\n")
    f.write(f"Final mAP50: {results_df['metrics/mAP50(B)'].max():.4f}\n")
    f.write("All graphs and weights successfully backed up.")

print(f"Professional backup for {CONDITION.upper()} completed successfully at: {BACKUP_DIR}")